# 02 — Merit order dispatch

**Does:** builds the merit order dispatch function, step by step.

**Input:** a toy 3-plant table first, then `data/raw/india_generator_stack.csv`

**Output:** a tested `merit_order_dispatch()` that will move to `src/dispatch.py`

**Method:** sort plants by marginal cost, dispatch the cheapest first until demand is met. The last plant running sets the clearing price. If demand can't be met, price = VOLL (scarcity price).

## Step 1: Toy example (3 plants)

Hand answers to check against:

| Demand | Dispatch | Price | Total cost |
|---|---|---|---|
| 30 MW | Solar 30 | 0 | 0 |
| 120 MW | Solar 40, Coal 80 | 2,000 | 160,000 |
| 200 MW | Solar 40, Coal 100, Gas 50, deficit 10 | VOLL | 425,000 |

In [ ]:
import pandas as pd

In [ ]:
plants = pd.DataFrame({
    "name": ["Gas", "Solar", "Coal"],
    "capacity_mw": [50, 40, 100],
    "cost": [4500, 0, 2000],
})
plants

In [ ]:
def merit_order_dispatch(demand, plants, voll=10000):
    # Arrange generators in increasing order of marginal cost
    plants = plants.sort_values("cost")

    remaining = demand
    total_cost = 0
    clearing_mc = None
    clearing_gen = None
    dispatch = {}

    for row in plants.itertuples():
        if remaining > row.capacity_mw:
            # Demand is bigger than this plant: run it at full capacity
            dispatch[row.name] = row.capacity_mw
            remaining -= row.capacity_mw
            total_cost += row.capacity_mw * row.cost
        else:
            # This plant covers what's left: it is the marginal plant
            dispatch[row.name] = remaining
            clearing_mc = row.cost
            clearing_gen = row.name
            total_cost += remaining * row.cost
            remaining = 0
            break

    surplus = plants["capacity_mw"].sum() - demand

    if remaining > 0:
        # Demand not met: scarcity price
        clearing_mc = voll
        clearing_gen = "Unserved"
        status = f"deficit of {remaining} MW"
    else:
        status = f"sufficient, surplus capacity {surplus} MW"

    return {
        "dispatch": dispatch,
        "clearing_gen": clearing_gen,
        "clearing_price": clearing_mc,
        "total_cost": total_cost,
        "status": status,
    }

In [ ]:
for d in [30, 120, 200]:
    print(d, merit_order_dispatch(d, plants))

## Step 2: Tests

The hand answers, plus boundary cases (demand exactly at a plant's limit). If anything breaks, an `assert` stops with an error.

In [ ]:
expected = {
    # demand: (price, total cost)
    30:  (0, 0),
    120: (2000, 160000),
    200: (10000, 425000),   # deficit -> VOLL
    0:   (0, 0),            # no demand
    40:  (0, 0),            # solar exactly full
    140: (2000, 200000),    # coal exactly full
    190: (4500, 425000),    # every plant exactly full, no deficit
}
for d, (price, cost) in expected.items():
    r = merit_order_dispatch(d, plants)
    assert r["clearing_price"] == price, f"{d} MW: price {r['clearing_price']}, expected {price}"
    assert r["total_cost"] == cost, f"{d} MW: cost {r['total_cost']}, expected {cost}"
print("All tests pass")

## Step 3 (your exercise): add a 4th plant

Add **Nuclear, 30 MW, Rs 3,500/MWh**. Before running, predict by hand what happens at 200 MW:
- Does the deficit disappear?
- What is the new clearing price, and which plant sets it?

Write your prediction here, then run the code and compare.

In [ ]:
# your code here